# Benchmark on T4: PyTorch vs ONNX (final detector weights)

## 0. Settings, packages, mount Drive

In [1]:
!pip -q install ultralytics onnxruntime-gpu

In [2]:
DRIVE_PROJECT = '/content/drive/MyDrive/AI251P_Capstone_CrossDomainAerial'
# final weights (P09_Model_Freeze.json: Joint_StrongerAug, 640) and their ONNX export (16 Nov)
DET_WEIGHTS = '03_Detection/2026-11-04_P08_Baseline/YOLO11n_Joint_StrongerAug/weights/best.pt'
DET_ONNX    = '03_Detection/Export/2026-11-16_YOLO11n_JointStrongerAug_v1/model.onnx'
MODEL_NAME  = 'YOLO11n_JointStrongerAug'
IMGSZ       = 640                       # stated input resolution: 640 x 640 letterbox, batch 1
BACKENDS    = ['pytorch_cuda', 'onnx_cuda']   # add 'onnx_cpu' for a CPU reference row
FRAMES_DIR  = '04_Segmentation/seg/yolo_dataset/images/val'   # real 1920x1080 frames
N_FRAMES, SEED = 50, 42
WARMUP, RUNS = 50, 500                  # warm-up runs are not timed
CONF, NMS_IOU = 0.25, 0.7               # NMS settings of the service (conf 0.25 not chosen on val)
BUDGET_P95_MS = 200                     # acceptance: p95 latency <= 200 ms per image on T4

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 1. Paths, frames, output folder

In [3]:
import os, re, sys, json, time, shutil, subprocess
from pathlib import Path
import numpy as np, pandas as pd

ROOT = Path(DRIVE_PROJECT)
P = lambda x: Path(x) if str(x).startswith('/') else ROOT / x
for name, p in (('weights', P(DET_WEIGHTS)), ('onnx', P(DET_ONNX))):
    if not p.is_file(): raise FileNotFoundError(f'{name} not found: {p} (set the path in section 0)')
GPU = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'],
                     capture_output=True, text=True).stdout.strip()
print('GPU:', GPU or 'none')
if 'T4' not in GPU: print('NOTE: not a T4; the acceptance criterion is stated for a T4 (Runtime > Change runtime type > T4 GPU)')

TODAY = time.strftime('%Y-%m-%d')
BDIR = ROOT / '03_Detection/Benchmark'; BDIR.mkdir(parents=True, exist_ok=True)
VERSION = max([int(m.group(1)) for f in os.listdir(BDIR) if (m := re.fullmatch(rf'{TODAY}_T4_v(\d+)', f))], default=0) + 1
OUT = BDIR / f'{TODAY}_T4_v{VERSION}'; OUT.mkdir()

LOCAL = Path('/content/bench'); shutil.rmtree(LOCAL, ignore_errors=True); (LOCAL / 'frames').mkdir(parents=True)
allf = sorted(p for p in P(FRAMES_DIR).rglob('*') if p.suffix.lower() in ('.jpg', '.jpeg', '.png'))
if not allf: raise FileNotFoundError(f'no frames under {P(FRAMES_DIR)}')
pick = [allf[i] for i in sorted(np.random.default_rng(SEED).choice(len(allf), min(N_FRAMES, len(allf)), replace=False))]
FRAMES = [str(shutil.copy2(f, LOCAL / 'frames' / f.name)) for f in pick]   # local copies: Drive reads are not timed
json.dump(FRAMES, open(LOCAL / 'frames.json', 'w'))
shutil.copy2(P(DET_WEIGHTS), LOCAL / 'best.pt'); shutil.copy2(P(DET_ONNX), LOCAL / 'model.onnx')
print(f'{len(FRAMES)} frames copied locally | output: {OUT.relative_to(ROOT)}')

GPU: Tesla T4, 15360 MiB, 580.82.07
30 frames copied locally | output: 03_Detection/Benchmark/2026-11-17_T4_v1


## 2. Benchmark script (`bench.py`)

In [4]:
BENCH_SRC = r"""# AI251P detector benchmark, one backend per process (clean GPU memory per backend).
# Per image (batch 1): preprocess (letterbox 640) -> inference -> NMS. Image decoding is excluded (frames decoded first).
import argparse, json, os, sys, time, threading, subprocess, resource
import numpy as np, cv2

def letterbox(img_bgr, size, color=114):
    h, w = img_bgr.shape[:2]; r = min(size / h, size / w)
    nh, nw = round(h * r), round(w * r)
    im = cv2.resize(img_bgr, (nw, nh), interpolation=cv2.INTER_LINEAR)
    top, left = (size - nh) // 2, (size - nw) // 2
    out = np.full((size, size, 3), color, np.uint8); out[top:top + nh, left:left + nw] = im
    return out

def to_input(img_bgr, size):   # BGR -> RGB, HWC -> CHW, /255, float32, [1,3,S,S]
    return np.ascontiguousarray(letterbox(img_bgr, size)[..., ::-1].transpose(2, 0, 1), dtype=np.float32)[None] / 255.0

def parse_apps(text, pid):
    # nvidia-smi --query-compute-apps=pid,used_memory --format=csv,noheader,nounits -> MiB used by pid
    tot = 0
    for line in text.strip().splitlines():
        parts = [p.strip() for p in line.split(',')]
        if len(parts) >= 2 and parts[0].isdigit() and int(parts[0]) == pid and parts[1].isdigit(): tot += int(parts[1])
    return tot

def stats(ms):
    a = np.asarray(ms, float)
    return {'p50_ms': round(float(np.percentile(a, 50)), 2), 'p95_ms': round(float(np.percentile(a, 95)), 2),
            'p99_ms': round(float(np.percentile(a, 99)), 2), 'mean_ms': round(float(a.mean()), 2), 'max_ms': round(float(a.max()), 2)}

class GpuMonitor(threading.Thread):
    def __init__(self, pid, every_s=0.25):
        super().__init__(daemon=True); self.pid, self.every, self.peak, self.stop = pid, every_s, 0, False
    def sample(self):
        try:
            t = subprocess.run(['nvidia-smi', '--query-compute-apps=pid,used_memory', '--format=csv,noheader,nounits'],
                               capture_output=True, text=True, timeout=5).stdout
            self.peak = max(self.peak, parse_apps(t, self.pid))
        except Exception: pass
    def run(self):
        while not self.stop: self.sample(); time.sleep(self.every)

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('--backend', required=True, choices=['pytorch_cuda', 'onnx_cuda', 'onnx_cpu', 'pytorch_cpu'])
    ap.add_argument('--weights'); ap.add_argument('--onnx'); ap.add_argument('--frames', required=True)
    ap.add_argument('--imgsz', type=int, default=640); ap.add_argument('--warmup', type=int, default=50)
    ap.add_argument('--runs', type=int, default=500); ap.add_argument('--conf', type=float, default=0.25)
    ap.add_argument('--iou', type=float, default=0.7); ap.add_argument('--out', required=True)
    a = ap.parse_args()
    paths = json.load(open(a.frames))
    frames = [cv2.imread(p) for p in paths]                 # decode first: excluded from timing
    frames = [f for f in frames if f is not None]
    if not frames: sys.exit('no readable frames')
    import torch
    try: from ultralytics.utils.nms import non_max_suppression
    except ImportError: from ultralytics.utils.ops import non_max_suppression
    mon = GpuMonitor(os.getpid()); mon.start()
    info = {'backend': a.backend, 'imgsz': a.imgsz, 'batch': 1, 'frames': len(frames), 'warmup': a.warmup, 'runs': a.runs,
            'conf': a.conf, 'nms_iou': a.iou, 'torch': torch.__version__}
    if a.backend.startswith('pytorch'):
        from ultralytics import YOLO
        import ultralytics; info['ultralytics'] = ultralytics.__version__
        dev = 'cuda' if a.backend == 'pytorch_cuda' else 'cpu'
        torch.backends.cudnn.benchmark = True
        model = YOLO(a.weights).model.float().eval().to(dev)
        if dev == 'cuda': torch.cuda.reset_peak_memory_stats()
        def infer(x):
            with torch.no_grad():
                y = model(torch.from_numpy(x).to(dev))
            return (y[0] if isinstance(y, (list, tuple)) else y).float().cpu()     # .cpu() waits for the GPU
        info['weights'] = a.weights
    else:
        import onnxruntime as ort; info['onnxruntime'] = ort.__version__
        prov = 'CUDAExecutionProvider' if a.backend == 'onnx_cuda' else 'CPUExecutionProvider'
        sess = ort.InferenceSession(a.onnx, providers=[prov])
        if sess.get_providers()[0] != prov: sys.exit(f'{prov} not available, session uses {sess.get_providers()}')
        name = sess.get_inputs()[0].name
        def infer(x): return torch.from_numpy(sess.run(None, {name: x})[0])
        info['onnx'] = a.onnx; info['provider'] = prov
    def nms(y): return non_max_suppression(y, conf_thres=a.conf, iou_thres=a.iou, max_det=300)[0]

    for i in range(a.warmup): nms(infer(to_input(frames[i % len(frames)], a.imgsz)))
    rows = []; t_start = time.perf_counter()
    for i in range(a.runs):
        img = frames[i % len(frames)]
        t0 = time.perf_counter(); x = to_input(img, a.imgsz)
        t1 = time.perf_counter(); y = infer(x)
        t2 = time.perf_counter(); d = nms(y)
        t3 = time.perf_counter()
        rows.append({'i': i, 'preprocess_ms': (t1 - t0) * 1e3, 'inference_ms': (t2 - t1) * 1e3, 'nms_ms': (t3 - t2) * 1e3,
                     'total_ms': (t3 - t0) * 1e3, 'detections': int(len(d))})
    wall = time.perf_counter() - t_start
    mon.sample(); mon.stop = True
    res = {**info,
           'latency_total': stats([r['total_ms'] for r in rows]), 'latency_inference': stats([r['inference_ms'] for r in rows]),
           'latency_preprocess': stats([r['preprocess_ms'] for r in rows]), 'latency_nms': stats([r['nms_ms'] for r in rows]),
           'throughput_img_per_s': round(a.runs / wall, 2),
           'throughput_inference_only_img_per_s': round(1000 / float(np.mean([r['inference_ms'] for r in rows])), 2),
           'gpu_mem_process_peak_mib': mon.peak if a.backend.endswith('cuda') else 0,
           'torch_peak_allocated_mib': round(torch.cuda.max_memory_allocated() / 2**20, 1) if a.backend == 'pytorch_cuda' else None,
           'host_rss_peak_mib': round(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024, 1),
           'mean_detections': round(float(np.mean([r['detections'] for r in rows])), 2)}
    json.dump({'result': res, 'per_image': rows}, open(a.out, 'w'), indent=1)
    print(json.dumps(res, indent=1))

if __name__ == '__main__': main()
"""

(LOCAL / 'bench.py').write_text(BENCH_SRC); (OUT / 'bench.py').write_text(BENCH_SRC)
print('bench.py written')

bench.py written


## 3. Run each backend in its own process

In [5]:
RES, PER = {}, {}
for b in BACKENDS:
    out = LOCAL / f'{b}.json'
    cmd = [sys.executable, str(LOCAL / 'bench.py'), '--backend', b, '--weights', str(LOCAL / 'best.pt'), '--onnx', str(LOCAL / 'model.onnx'),
           '--frames', str(LOCAL / 'frames.json'), '--imgsz', str(IMGSZ), '--warmup', str(WARMUP), '--runs', str(RUNS),
           '--conf', str(CONF), '--iou', str(NMS_IOU), '--out', str(out)]
    print(f'=== {b} ==='); t = time.time()
    p = subprocess.run(cmd, capture_output=True, text=True)
    if p.returncode != 0: print(p.stdout[-2000:], p.stderr[-3000:]); raise RuntimeError(f'{b} failed')
    d = json.load(open(out)); RES[b], PER[b] = d['result'], pd.DataFrame(d['per_image'])
    PER[b].to_csv(OUT / f'per_image_{b}.csv', index=False)
    r = RES[b]; print(f"p50 {r['latency_total']['p50_ms']} | p95 {r['latency_total']['p95_ms']} | p99 {r['latency_total']['p99_ms']} ms | "
                      f"{r['throughput_img_per_s']} img/s | GPU mem {r['gpu_mem_process_peak_mib']} MiB ({time.time() - t:.0f}s)")

=== pytorch_cuda ===
p50 20.85 | p95 34.72 | p99 50.1 ms | 42.42 img/s | GPU mem 242 MiB (30s)
=== onnx_cuda ===
p50 13.69 | p95 17.84 | p99 19.66 ms | 70.58 img/s | GPU mem 188 MiB (13s)


## 4. Benchmark table and acceptance

In [6]:
LABEL = {'pytorch_cuda': 'PyTorch fp32 (CUDA)', 'onnx_cuda': 'ONNX Runtime fp32 (CUDA EP)', 'onnx_cpu': 'ONNX Runtime fp32 (CPU)', 'pytorch_cpu': 'PyTorch fp32 (CPU)'}
rows = []
for b, r in RES.items():
    T, I = r['latency_total'], r['latency_inference']
    rows.append({'Backend': LABEL[b], 'Input': f'{IMGSZ}x{IMGSZ}, batch 1',
                 'p50 ms': T['p50_ms'], 'p95 ms': T['p95_ms'], 'p99 ms': T['p99_ms'],
                 'Inference only p50 / p95 ms': f"{I['p50_ms']} / {I['p95_ms']}",
                 'Throughput img/s': r['throughput_img_per_s'],
                 'GPU memory peak MiB': r['gpu_mem_process_peak_mib'] or '-',
                 'Host RAM peak MiB': r['host_rss_peak_mib'],
                 f'p95 <= {BUDGET_P95_MS} ms': 'MET' if T['p95_ms'] <= BUDGET_P95_MS else 'NOT MET'})
TABLE = pd.DataFrame(rows)
TABLE.to_csv(OUT / 'benchmark_table.csv', index=False)
print(TABLE.to_string(index=False))

GPU_ROWS = {b: r for b, r in RES.items() if b.endswith('cuda')}
ACCEPT = {'criterion': f'p95 latency <= {BUDGET_P95_MS} ms per image on T4 (preprocess + inference + NMS, batch 1, decode excluded)',
          'gpu': GPU, 'per_backend': {b: {'p95_ms': r['latency_total']['p95_ms'], 'met': r['latency_total']['p95_ms'] <= BUDGET_P95_MS} for b, r in GPU_ROWS.items()}}
ACCEPT['met'] = bool(GPU_ROWS) and all(v['met'] for v in ACCEPT['per_backend'].values()) and 'T4' in GPU
print('\nAcceptance:', 'MET' if ACCEPT['met'] else 'NOT MET', ACCEPT['per_backend'])

                    Backend            Input  p50 ms  p95 ms  p99 ms Inference only p50 / p95 ms  Throughput img/s  GPU memory peak MiB  Host RAM peak MiB p95 <= 200 ms
        PyTorch fp32 (CUDA) 640x640, batch 1   20.85   34.72   50.10               13.03 / 22.61             42.42                  242             1442.3           MET
ONNX Runtime fp32 (CUDA EP) 640x640, batch 1   13.69   17.84   19.66                 6.41 / 7.21             70.58                  188             1648.6           MET

Acceptance: MET {'pytorch_cuda': {'p95_ms': 34.72, 'met': True}, 'onnx_cuda': {'p95_ms': 17.84, 'met': True}}


## 5. Report

In [7]:
import onnxruntime, ultralytics, torch
REPORT = {'created': time.strftime('%Y-%m-%d %H:%M'), 'output': OUT.relative_to(ROOT).as_posix(), 'model': MODEL_NAME,
          'weights': DET_WEIGHTS, 'onnx': DET_ONNX, 'final_weights_source': 'P09_Model_Freeze.json (primary_model Joint_StrongerAug)',
          'input_resolution': f'{IMGSZ}x{IMGSZ} letterbox (pad 114), batch 1, fp32',
          'measured': 'per image: preprocess + inference + NMS (conf %.2f, IoU %.2f); image decoding excluded' % (CONF, NMS_IOU),
          'frames': {'dir': FRAMES_DIR, 'n': len(FRAMES), 'seed': SEED, 'size': '1920x1080'}, 'warmup': WARMUP, 'runs': RUNS,
          'memory': 'GPU: peak memory of the benchmark process from nvidia-smi (includes the CUDA context); host: peak RSS',
          'gpu': GPU, 'versions': {'torch': torch.__version__, 'ultralytics': ultralytics.__version__, 'onnxruntime': onnxruntime.__version__},
          'results': RES, 'acceptance': ACCEPT}
(OUT / 'benchmark_report.json').write_text(json.dumps(REPORT, indent=1, default=str))
try: md_rows = TABLE.to_markdown(index=False)          # needs tabulate
except ImportError: md_rows = '```\n' + TABLE.to_string(index=False) + '\n```'
(OUT / 'README.md').write_text(f"""# Detector benchmark on T4: {MODEL_NAME} ({TODAY}, v{VERSION})

Final weights `{DET_WEIGHTS}` and ONNX export `{DET_ONNX}`. GPU: {GPU}.
Input resolution: **{IMGSZ}x{IMGSZ}** letterbox, batch 1, fp32. Per image: preprocess + inference + NMS; decoding excluded.
{len(FRAMES)} real 1920x1080 frames, {WARMUP} warm-up runs (not timed), {RUNS} timed runs per backend, each backend in its own process.

{md_rows}

**Acceptance (p95 <= {BUDGET_P95_MS} ms per image on T4): {'MET' if ACCEPT['met'] else 'NOT MET'}**

Scope: the detector only. The full integration pipeline (detector + classifier + segmentation) is reported separately.
GPU memory is the peak used by the benchmark process (nvidia-smi), including the CUDA context.
""")
print(open(OUT / 'README.md').read())
print('files:', sorted(p.name for p in OUT.iterdir()))

# Detector benchmark on T4: YOLO11n_JointStrongerAug (2026-11-17, v1)

Final weights `03_Detection/2026-11-04_P08_Baseline/YOLO11n_Joint_StrongerAug/weights/best.pt` and ONNX export `03_Detection/Export/2026-11-16_YOLO11n_JointStrongerAug_v1/model.onnx`. GPU: Tesla T4, 15360 MiB, 580.82.07.
Input resolution: **640x640** letterbox, batch 1, fp32. Per image: preprocess + inference + NMS; decoding excluded.
30 real 1920x1080 frames, 50 warm-up runs (not timed), 500 timed runs per backend, each backend in its own process.

| Backend                     | Input            |   p50 ms |   p95 ms |   p99 ms | Inference only p50 / p95 ms   |   Throughput img/s |   GPU memory peak MiB |   Host RAM peak MiB | p95 <= 200 ms   |
|:----------------------------|:-----------------|---------:|---------:|---------:|:------------------------------|-------------------:|----------------------:|--------------------:|:----------------|
| PyTorch fp32 (CUDA)         | 640x640, batch 1 |    20.85 |    34.72 | 